Folder Upload

In [ ]:
from google.colab import files
upoaded = files.upload()

Saving submission.csv to submission.csv
Saving test.csv to test.csv
Saving train.csv to train.csv


In [ ]:
import pandas as pd
url = 'https://raw.githubusercontent.com/dhananjay-deshmukh/flight-price-prediction/refs/heads/main/data/train.csv?token=GHSAT0AAAAAAEMBBM6526RLURSMLA37BSVO2WLZRAA'
df = pd.read_csv(url)
df.head()

(18076, 12)
(4520, 11)
(4520, 2)


In [ ]:
# Column names and data types
train.info()

# Missing values
print("Missing values:\n", train.isnull().sum())

# Duplicate records
print("Duplicate rows:", train.duplicated().sum())

# Price statistics
display(train["price"].describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18076 entries, 0 to 18075
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                18076 non-null  int64  
 1   airline           18076 non-null  object 
 2   flight            18076 non-null  object 
 3   source_city       18076 non-null  object 
 4   departure_time    18076 non-null  object 
 5   stops             18076 non-null  object 
 6   arrival_time      18076 non-null  object 
 7   destination_city  18076 non-null  object 
 8   flight_class      18076 non-null  object 
 9   duration          18076 non-null  float64
 10  days_left         18076 non-null  int64  
 11  price             18076 non-null  int64  
dtypes: float64(1), int64(3), object(8)
memory usage: 1.7+ MB
Missing values:
 id                  0
airline             0
flight              0
source_city         0
departure_time      0
stops               0
arrival_time        0


,price
count,18076.000000
mean,280.264550
std,221.965133
min,0.000000
25%,157.000000
50%,234.000000
75%,336.000000
max,8996.000000


In [ ]:
# Separate features and target
X = train.drop(columns=["price", "id", "flight"], errors="ignore")
y = train["price"]

# Separate test features
X_test = test.drop(columns=["price", "id", "flight"], errors="ignore")

# Ensure the columns match
X_test = X_test[X.columns]

print("Training features:", X.shape)
print("Target:", y.shape)
print("Test features:", X_test.shape)

Training features: (18076, 9)
Target: (18076,)
Test features: (4520, 9)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Training rows:", len(X_train))
print("Validation rows:", len(X_val))

Training rows: 14460
Validation rows: 3616


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = X.select_dtypes(include=["number"]).columns.tolist()

preprocessor = ColumnTransformer([
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_cols),

    ("numerical", Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ]), numerical_cols)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

model.fit(X_train, y_train)

print("Model training completed!")

Model training completed!


In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

val_predictions = model.predict(X_val)

mae = mean_absolute_error(y_val, val_predictions)
rmse = np.sqrt(mean_squared_error(y_val, val_predictions))
r2 = r2_score(y_val, val_predictions)

print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R² score:", round(r2, 4))

MAE: 47.41
RMSE: 148.26
R² score: 0.6092


Extra trees regressor

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

extra_trees = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("regressor", ExtraTreesRegressor(
        n_estimators=200,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

extra_trees.fit(X_train, y_train)

pred = extra_trees.predict(X_val)

print("Extra Trees Results")
print("MAE:", round(mean_absolute_error(y_val, pred), 2))
print("RMSE:", round(np.sqrt(mean_squared_error(y_val, pred)), 2))
print("R²:", round(r2_score(y_val, pred), 4))

Extra Trees Results
MAE: 46.03
RMSE: 141.39
R²: 0.6446


After extra

In [ ]:
import pandas as pd
from google.colab import files

# Rebuild submission using the actual test data
submission = submission.copy()
submission["id"] = test["id"].values
submission["price"] = extra_trees.fit(X, y).predict(X_test)

# Save
submission.to_csv("submission.csv", index=False)

# Verify against the actual dataset and template
check = pd.read_csv("submission.csv")

assert check.shape == (len(test), 2)
assert check.columns.tolist() == submission.columns.tolist()
assert check["id"].equals(test["id"])
assert check["price"].notna().all()

print("All checks passed!")
print("Final shape:", check.shape)
print(check.head())

files.download("submission.csv")

All checks passed!
Final shape: (4520, 2)
      id       price
0  18395   62.463929
1  11771  157.827500
2  22326  372.190780
3  18332  142.438333
4  11021  559.170932


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Tune trees.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.pipeline import Pipeline
from sklearn.base import clone

pipeline = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("regressor", ExtraTreesRegressor(
        random_state=42,
        n_jobs=-1
    ))
])

param_dist = {
    "regressor__n_estimators": [200, 300, 500],
    "regressor__max_depth": [None, 15, 25, 35],
    "regressor__min_samples_split": [2, 5, 10],
    "regressor__min_samples_leaf": [1, 2, 3],
    "regressor__max_features": [0.7, 0.9, 1.0]
}

search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_dist,
    n_iter=15,
    scoring="neg_root_mean_squared_error",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

search.fit(X, y)

print("Best CV RMSE:", -search.best_score_)
print("Best parameters:", search.best_params_)

Fitting 3 folds for each of 15 candidates, totalling 45 fits


/usr/local/lib/python3.13/dist-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best CV RMSE: 124.7807780155703
Best parameters: {'regressor__n_estimators': 500, 'regressor__min_samples_split': 10, 'regressor__min_samples_leaf': 2, 'regressor__max_features': 0.7, 'regressor__max_depth': 25}


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

best_model = search.best_estimator_

# Evaluate on the same validation data as before
val_pred = best_model.predict(X_val)

print("Tuned Extra Trees")
print("MAE:", round(mean_absolute_error(y_val, val_pred), 2))
print("RMSE:", round(np.sqrt(mean_squared_error(y_val, val_pred)), 2))
print("R²:", round(r2_score(y_val, val_pred), 4))

Tuned Extra Trees
MAE: 35.75
RMSE: 110.66
R²: 0.7823


In [ ]:
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.pipeline import Pipeline
from google.colab import files

# Build the tuned model directly
final_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("regressor", ExtraTreesRegressor(
        n_estimators=500,
        max_depth=25,
        min_samples_split=10,
        min_samples_leaf=2,
        max_features=0.7,
        random_state=42,
        n_jobs=-1
    ))
])

# Train once on the complete training dataset
final_model.fit(X, y)

# Generate test predictions
test_predictions = final_model.predict(X_test)

# Preserve the required submission structure
submission =submission.copy()
submission["id"] = test["id"].values
submission["price"] = test_predictions

submission.to_csv("submission.csv", index=False)

assert submission.shape == (len(test), 2)
assert submission.columns.tolist() == ["id", "price"]
assert submission["id"].equals(test["id"])
assert submission["price"].notna().all()

print("Submission ready!")
print("Shape:", submission.shape)

files.download("submission.csv")

Submission ready!
Shape: (4520, 2)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Trying Catboot

In [ ]:
!pip -q install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.3 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Identify categorical columns from the original features
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

# CatBoost accepts categorical values as strings
X_cb_train = X_train.copy()
X_cb_val = X_val.copy()

for col in cat_cols:
    X_cb_train[col] = X_cb_train[col].fillna("Unknown").astype(str)
    X_cb_val[col] = X_cb_val[col].fillna("Unknown").astype(str)

# Train the model
cat_model = CatBoostRegressor(
    iterations=1200,
    depth=8,
    learning_rate=0.05,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42,
    verbose=200,
    thread_count=2
)

cat_model.fit(
    X_cb_train,
    y_train,
    cat_features=cat_cols,
    eval_set=(X_cb_val, y_val),
    early_stopping_rounds=100,
    verbose=200
)

# Evaluate on validation data
val_pred = cat_model.predict(X_cb_val)

print("\nCatBoost validation results")
print("MAE:", round(mean_absolute_error(y_val, val_pred), 2))
print("RMSE:", round(np.sqrt(mean_squared_error(y_val, val_pred)), 2))
print("R²:", round(r2_score(y_val, val_pred), 4))
print("Best iteration:", cat_model.best_iteration_)

0:	learn: 212.6035507	test: 232.1213473	best: 232.1213473 (0)	total: 85.9ms	remaining: 1m 43s
200:	learn: 106.0207123	test: 151.6355094	best: 151.6355094 (200)	total: 3.21s	remaining: 16s
400:	learn: 94.5020530	test: 145.7029438	best: 145.7029438 (400)	total: 8.62s	remaining: 17.2s
600:	learn: 88.0241207	test: 143.6845417	best: 143.6845417 (600)	total: 13.9s	remaining: 13.8s
800:	learn: 83.0129224	test: 142.6811408	best: 142.6760760 (783)	total: 20.8s	remaining: 10.4s
1000:	learn: 78.7923162	test: 141.9817807	best: 141.9817807 (1000)	total: 26.6s	remaining: 5.29s
1199:	learn: 75.8377721	test: 141.4815430	best: 141.4815430 (1199)	total: 33.8s	remaining: 0us

bestTest = 141.481543
bestIteration = 1199


CatBoost validation results
MAE: 49.31
RMSE: 141.48
R²: 0.6442
Best iteration: 1199


In [ ]:
from catboost import CatBoostRegressor
import pandas as pd
from google.colab import files

# Prepare all training features
X_full = X.copy()
X_test_full = X_test.copy()

cat_cols = X_full.select_dtypes(
    include=["object", "category"]
).columns.tolist()

for col in cat_cols:
    X_full[col] = X_full[col].fillna("Unknown").astype(str)
    X_test_full[col] = X_test_full[col].fillna("Unknown").astype(str)

# Train final CatBoost model
final_cat = CatBoostRegressor(
    iterations=1200,
    depth=8,
    learning_rate=0.05,
    loss_function="RMSE",
    random_seed=42,
    verbose=200,
    thread_count=2
)

final_cat.fit(
    X_full,
    y,
    cat_features=cat_cols
)

# Predict test prices
cat_predictions = final_cat.predict(X_test_full)

print("Predictions generated:", len(cat_predictions))

0:	learn: 216.8582522	total: 185ms	remaining: 3m 41s
200:	learn: 109.9212366	total: 6.71s	remaining: 33.3s
400:	learn: 97.6563761	total: 13.2s	remaining: 26.4s
600:	learn: 91.4720529	total: 19.4s	remaining: 19.3s
800:	learn: 87.2866612	total: 26.5s	remaining: 13.2s
1000:	learn: 84.0278283	total: 32.8s	remaining: 6.52s
1199:	learn: 80.7689304	total: 40.5s	remaining: 0us
Predictions generated: 4520


In [ ]:
submission_cat = submission.copy()

submission_cat["id"] = test["id"].values
submission_cat["price"] = cat_predictions

submission_cat.to_csv("catboost_submission.csv", index=False)

# Verify the required format
check = pd.read_csv("catboost_submission.csv")

assert check.shape == (len(test), 2)
assert check.columns.tolist() == ["id", "price"]
assert check["id"].equals(test["id"])
assert check["price"].notna().all()

print("Submission shape:", check.shape)
display(check.head())

Submission shape: (4520, 2)


,id,price
0,18395,69.268819
1,11771,176.334115
2,22326,438.233297
3,18332,121.248896
4,11021,540.051520


In [ ]:
files.download("catboost_submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Dense output instead of sparse output
preprocessor_dense = ColumnTransformer([
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ]), categorical_cols),

    ("numerical", Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ]), numerical_cols)
])

boosted_model = Pipeline([
    ("preprocessor", preprocessor_dense),
    ("regressor", HistGradientBoostingRegressor(
        max_iter=200,
        learning_rate=0.1,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=42
    ))
])

boosted_model.fit(X_train, y_train)

pred = boosted_model.predict(X_val)

print("MAE:", round(mean_absolute_error(y_val, pred), 2))
print("RMSE:", round(np.sqrt(mean_squared_error(y_val, pred)), 2))
print("R²:", round(r2_score(y_val, pred), 4))

MAE: 53.46
RMSE: 145.75
R²: 0.6224
